In [1]:
"""
rag_ingestion.py  —  Document Ingestion Pipeline for RAG
─────────────────────────────────────────────────────────
Supports: .docx, .pdf
Output:
  - Console logs of parent chunks
  - parent_chunks.txt
  - child_chunks.txt
  - All content stored as Markdown (for LLM readability)

Usage:
    python rag_ingestion.py path/to/your/file.docx
    python rag_ingestion.py path/to/your/file.pdf
"""

import re
import sys
import os
from collections import Counter
from dataclasses import dataclass, field


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

PARENT_MAX_TOKENS = 800     # split parent if section exceeds this
CHILD_MAX_TOKENS  = 150     # target size for child chunks
CHILD_OVERLAP     = 30      # token overlap between consecutive children

OUTPUT_PARENTS = "parent_chunks.txt"
OUTPUT_CHILDREN = "child_chunks.txt"


# ─────────────────────────────────────────────────────────────────────────────
# TOKEN ESTIMATOR  (approx: 1 token ≈ 4 chars)
# ─────────────────────────────────────────────────────────────────────────────

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)


# ─────────────────────────────────────────────────────────────────────────────
# COMMON ELEMENT SCHEMA
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Element:
    type: str          # title | h1 | h2 | h3 | body | list_item | code | table | caption
    text: str          # raw text content
    page: int = 0      # page number (PDF) or 0 (docx)


# ─────────────────────────────────────────────────────────────────────────────
# FORMAT DETECTION
# ─────────────────────────────────────────────────────────────────────────────

def detect_format(path: str) -> str:
    """Detect format by extension first, then magic bytes as fallback."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".docx":
        return "docx"
    if ext == ".pdf":
        return "pdf"

    # Magic bytes fallback
    with open(path, "rb") as f:
        header = f.read(8)
    if header[:4] == b"%PDF":
        return "pdf"
    if header[:4] == b"PK\x03\x04":   # ZIP = docx/xlsx/pptx
        return "docx"

    raise ValueError(f"Unsupported format: {ext} — only .docx and .pdf supported")


# ─────────────────────────────────────────────────────────────────────────────
# DOCX ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def _size_pt(runs):
    sizes = [r.font.size for r in runs if r.font.size]
    return round(max(sizes) / 12700) if sizes else None

def _all_bold(runs):
    tr = [r for r in runs if r.text.strip()]
    return bool(tr) and all(r.bold for r in tr)

def _all_italic(runs):
    tr = [r for r in runs if r.text.strip()]
    return bool(tr) and all(r.italic for r in tr)

def _build_size_map(raw_elements: list) -> dict:
    size_counts = Counter()
    total = 0
    for el in raw_elements:
        sz = el.get("size_pt")
        if sz:
            size_counts[sz] += 1
            total += 1
    if not size_counts:
        return {}
    body_threshold = max(total * 0.25, 2)
    body_sizes     = {sz for sz, cnt in size_counts.items() if cnt >= body_threshold}
    heading_sizes  = sorted([sz for sz in size_counts if sz not in body_sizes], reverse=True)
    size_map = {sz: "body" for sz in body_sizes}
    for i, sz in enumerate(heading_sizes):
        size_map[sz] = ["title", "h1", "h2", "h3"][min(i, 3)]
    return size_map

STYLE_ROLE = {
    "heading 1": "h1", "heading 2": "h2", "heading 3": "h3", "heading 4": "h3",
    "title": "title", "subtitle": "caption",
    "list paragraph": "list_item", "list bullet": "list_item", "list number": "list_item",
    "caption": "caption", "intense quote": "body", "quote": "body",
}

def extract_docx(path: str) -> list[Element]:
    from docx import Document

    doc      = Document(path)
    body     = doc.element.body
    para_map = {p._element: p for p in doc.paragraphs}
    tbl_map  = {t._element: t for t in doc.tables}

    # First pass: collect raw signals for size calibration
    raw = []
    for child in body:
        tag = child.tag.split('}')[-1]
        if tag == 'p':
            para = para_map.get(child)
            if para and para.text.strip():
                raw.append({"size_pt": _size_pt(para.runs)})
        elif tag == 'tbl':
            tbl = tbl_map.get(child)
            if tbl:
                for row in tbl.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            if p.text.strip():
                                raw.append({"size_pt": _size_pt(p.runs)})
                                break

    size_map = _build_size_map(raw)

    # Second pass: classify and emit elements
    elements = []

    def classify_para(para):
        style_lower = para.style.name.lower() if para.style else ""
        for key, role in STYLE_ROLE.items():
            if key in style_lower:
                return role
        size  = _size_pt(para.runs)
        bold  = _all_bold(para.runs)
        italic = _all_italic(para.runs)
        if size and size in size_map:
            level = size_map[size]
            if level in ("title", "h1", "h2", "h3"):
                return level
        if bold:
            return "h3"
        text_lower = para.text.lower()
        if italic or any(kw in text_lower for kw in ("next:", "complete.", "coming up")):
            return "caption"
        return "body"

    def classify_table(tbl):
        nrows = len(tbl.rows)
        ncols = len(tbl.columns) if tbl.rows else 0
        first_text = first_size = None
        first_bold = False
        for row in tbl.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    t = p.text.strip()
                    if t:
                        first_text = t
                        first_size = _size_pt(p.runs)
                        first_bold = _all_bold(p.runs)
                        break
                if first_text:
                    break
            if first_text:
                break
        if nrows > 2 and ncols >= 2:
            return "table"
        if ncols <= 1:
            if not first_bold:
                return "code"
            has_emoji  = first_text and any(ord(c) > 0x2000 for c in first_text[:3])
            size_level = size_map.get(first_size, "body") if first_size else "body"
            is_banner  = first_text and len(first_text) < 80 and not has_emoji and (
                first_text.upper() == first_text
                or (first_text[:1].isdigit())
                or size_level in ("title", "h1", "h2")
            )
            return "h1" if is_banner else "body"
        return "table"

    def table_to_markdown(tbl) -> str:
        rows = []
        for row in tbl.rows:
            cells = [" ".join(p.text.strip() for p in cell.paragraphs if p.text.strip())
                     for cell in row.cells]
            rows.append(cells)
        if not rows:
            return ""
        ncols = max(len(r) for r in rows)
        lines = []
        header = rows[0]
        lines.append("| " + " | ".join(c.ljust(1) for c in header) + " |")
        lines.append("| " + " | ".join(["---"] * ncols) + " |")
        for row in rows[1:]:
            padded = row + [""] * (ncols - len(row))
            lines.append("| " + " | ".join(padded) + " |")
        return "\n".join(lines)

    def code_to_markdown(tbl) -> str:
        lines = []
        for row in tbl.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    if p.text.strip():
                        lines.append(p.text)
        return "```python\n" + "\n".join(lines) + "\n```"

    for child in body:
        tag = child.tag.split('}')[-1]

        if tag == 'p':
            para = para_map.get(child)
            if not para or not para.text.strip():
                continue
            role = classify_para(para)
            elements.append(Element(type=role, text=para.text.strip(), page=0))

        elif tag == 'tbl':
            tbl = tbl_map.get(child)
            if not tbl:
                continue
            role = classify_table(tbl)
            if role == "code":
                md = code_to_markdown(tbl)
                elements.append(Element(type="code", text=md, page=0))
            elif role == "table":
                md = table_to_markdown(tbl)
                elements.append(Element(type="table", text=md, page=0))
            else:
                # Banner/heading stored as h1 with its text
                first_text = ""
                for row in tbl.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            if p.text.strip():
                                first_text = p.text.strip()
                                break
                        if first_text:
                            break
                    if first_text:
                        break
                if first_text:
                    elements.append(Element(type=role, text=first_text, page=0))

    return elements


# ─────────────────────────────────────────────────────────────────────────────
# PDF ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def extract_pdf(path: str) -> list[Element]:
    import fitz  # PyMuPDF

    doc      = fitz.open(path)
    raw_paras = []   # {text, size, bold, page}

    for page_num, page in enumerate(doc, start=1):
        blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]
        for block in blocks:
            if block.get("type") != 0:   # 0 = text block
                continue
            for line in block.get("lines", []):
                line_text  = ""
                line_size  = 0
                line_bold  = False
                for span in line.get("spans", []):
                    t = span.get("text", "").strip()
                    if not t:
                        continue
                    line_text += t + " "
                    line_size  = max(line_size, span.get("size", 0))
                    flags      = span.get("flags", 0)
                    line_bold  = line_bold or bool(flags & 2**4)  # bold flag
                line_text = line_text.strip()
                if line_text:
                    raw_paras.append({
                        "text":    line_text,
                        "size":    round(line_size),
                        "bold":    line_bold,
                        "page":    page_num,
                    })

    if not raw_paras:
        return []

    # Build size map (same relative-ranking logic as docx)
    size_counts = Counter(p["size"] for p in raw_paras if p["size"])
    total       = len(raw_paras)
    body_threshold = max(total * 0.25, 3)
    body_sizes  = {sz for sz, cnt in size_counts.items() if cnt >= body_threshold}
    heading_sizes = sorted([sz for sz in size_counts if sz not in body_sizes], reverse=True)
    size_map = {sz: "body" for sz in body_sizes}
    for i, sz in enumerate(heading_sizes):
        size_map[sz] = ["title", "h1", "h2", "h3"][min(i, 3)]

    elements = []
    for p in raw_paras:
        size = p["size"]
        bold = p["bold"]
        text = p["text"]

        level = size_map.get(size, "body")
        if level in ("title", "h1", "h2", "h3"):
            role = level
        elif bold:
            role = "h3"
        else:
            role = "body"

        elements.append(Element(type=role, text=text, page=p["page"]))

    return elements


# ─────────────────────────────────────────────────────────────────────────────
# TITLE EXTRACTION  (from content, not filename)
# ─────────────────────────────────────────────────────────────────────────────

def extract_title(elements: list[Element]) -> str:
    """Return the first title/h1 element's text as the document title."""
    for el in elements[:10]:   # title is always near the top
        if el.type in ("title", "h1"):
            return el.text
    # Fallback: first non-empty text
    for el in elements:
        if el.text.strip():
            return el.text.strip()
    return "Untitled Document"


# ─────────────────────────────────────────────────────────────────────────────
# ELEMENT → MARKDOWN
# ─────────────────────────────────────────────────────────────────────────────

HEADING_MD = {
    "title": "# ",
    "h1":    "# ",
    "h2":    "## ",
    "h3":    "### ",
}

def element_to_markdown(el: Element) -> str:
    if el.type in HEADING_MD:
        return HEADING_MD[el.type] + el.text
    if el.type == "list_item":
        return "- " + el.text
    if el.type in ("code",):
        # Already formatted as markdown fenced block by engine
        return el.text if el.text.startswith("```") else f"```\n{el.text}\n```"
    if el.type == "table":
        return el.text   # Already markdown table
    if el.type == "caption":
        return f"*{el.text}*"
    return el.text   # body


# ─────────────────────────────────────────────────────────────────────────────
# PARENT CHUNKING
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class ParentChunk:
    parent_id:    str
    doc_title:    str
    heading_path: list[str]
    markdown:     str
    page_start:   int
    token_count:  int


def build_parent_chunks(elements: list[Element], doc_title: str) -> list[ParentChunk]:
    """
    Group elements into sections by heading boundaries.
    If a section exceeds PARENT_MAX_TOKENS, split it into overlapping sub-chunks.
    Each chunk is stored as Markdown so the LLM receives structured context.
    """
    HEADING_TYPES = {"title", "h1", "h2", "h3"}

    # Track current heading path at each level
    heading_path = []
    current_section_elements: list[Element] = []
    current_page = 0
    all_sections = []   # list of (heading_path_snapshot, elements_list, page_start)

    def flush_section():
        if current_section_elements:
            all_sections.append((
                list(heading_path),
                list(current_section_elements),
                current_page,
            ))

    for el in elements:
        if el.type in HEADING_TYPES:
            flush_section()
            current_section_elements = [el]
            current_page = el.page

            # Update heading path
            level = {"title": 0, "h1": 1, "h2": 2, "h3": 3}[el.type]
            heading_path = heading_path[:level]
            heading_path.append(el.text)
        else:
            current_section_elements.append(el)
            if el.page and not current_page:
                current_page = el.page

    flush_section()

    # Convert sections to markdown chunks, splitting oversized ones
    chunks      = []
    chunk_index = 0

    def make_chunk(h_path, md_text, page, idx):
        return ParentChunk(
            parent_id    = f"parent_{idx:04d}",
            doc_title    = doc_title,
            heading_path = h_path,
            markdown     = md_text,
            page_start   = page,
            token_count  = estimate_tokens(md_text),
        )

    for h_path, sec_elements, page in all_sections:
        # Convert each element to markdown
        md_lines = [element_to_markdown(el) for el in sec_elements]
        full_md  = "\n\n".join(line for line in md_lines if line.strip())

        if estimate_tokens(full_md) <= PARENT_MAX_TOKENS:
            chunks.append(make_chunk(h_path, full_md, page, chunk_index))
            chunk_index += 1
        else:
            # Split by paragraphs with overlap
            paragraphs  = [element_to_markdown(el) for el in sec_elements if element_to_markdown(el).strip()]
            buf         = []
            buf_tokens  = 0
            overlap_buf = []

            for para in paragraphs:
                para_tokens = estimate_tokens(para)

                if buf_tokens + para_tokens > PARENT_MAX_TOKENS and buf:
                    # Emit current buffer as a chunk
                    chunk_md = "\n\n".join(buf)
                    chunks.append(make_chunk(h_path, chunk_md, page, chunk_index))
                    chunk_index += 1

                    # Keep last few paragraphs as overlap for next chunk
                    overlap_tokens = 0
                    overlap_buf    = []
                    for p in reversed(buf):
                        pt = estimate_tokens(p)
                        if overlap_tokens + pt > CHILD_OVERLAP * 4:
                            break
                        overlap_buf.insert(0, p)
                        overlap_tokens += pt

                    buf        = overlap_buf + [para]
                    buf_tokens = sum(estimate_tokens(p) for p in buf)
                else:
                    buf.append(para)
                    buf_tokens += para_tokens

            if buf:
                chunk_md = "\n\n".join(buf)
                chunks.append(make_chunk(h_path, chunk_md, page, chunk_index))
                chunk_index += 1

    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# CHILD CHUNKING
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class ChildChunk:
    child_id:     str
    parent_id:    str
    doc_title:    str
    heading_path: list[str]
    markdown:     str
    page_start:   int
    token_count:  int


def split_into_sentences(text: str) -> list[str]:
    """Naive but effective sentence splitter."""
    # Don't split code blocks or tables
    if text.startswith("```") or text.startswith("|"):
        return [text]
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def build_child_chunks(parent_chunks: list[ParentChunk]) -> list[ChildChunk]:
    """
    Split each parent chunk into child chunks of ~CHILD_MAX_TOKENS.
    Code blocks and tables are kept as single children (not split).
    Consecutive text sentences are grouped until token limit is hit.
    """
    children    = []
    child_index = 0

    for parent in parent_chunks:
        lines = parent.markdown.split("\n\n")

        sentence_buffer  = []
        buffer_tokens    = 0

        def flush_buffer():
            nonlocal child_index, sentence_buffer, buffer_tokens
            if not sentence_buffer:
                return
            md = " ".join(sentence_buffer)
            children.append(ChildChunk(
                child_id     = f"child_{child_index:05d}",
                parent_id    = parent.parent_id,
                doc_title    = parent.doc_title,
                heading_path = parent.heading_path,
                markdown     = md,
                page_start   = parent.page_start,
                token_count  = estimate_tokens(md),
            ))
            child_index    += 1
            sentence_buffer = []
            buffer_tokens   = 0

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Code blocks and tables → single child each, never split
            if line.startswith("```") or line.startswith("|"):
                flush_buffer()
                children.append(ChildChunk(
                    child_id     = f"child_{child_index:05d}",
                    parent_id    = parent.parent_id,
                    doc_title    = parent.doc_title,
                    heading_path = parent.heading_path,
                    markdown     = line,
                    page_start   = parent.page_start,
                    token_count  = estimate_tokens(line),
                ))
                child_index += 1
                continue

            # Headings go into their own micro-chunk to anchor context
            if line.startswith("#"):
                flush_buffer()
                sentence_buffer = [line]
                buffer_tokens   = estimate_tokens(line)
                continue

            # Regular text → split into sentences, group into child chunks
            sentences = split_into_sentences(line)
            for sent in sentences:
                sent_tokens = estimate_tokens(sent)
                if buffer_tokens + sent_tokens > CHILD_MAX_TOKENS and sentence_buffer:
                    flush_buffer()
                sentence_buffer.append(sent)
                buffer_tokens += sent_tokens

        flush_buffer()

    return children


# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

def print_and_save(parents: list[ParentChunk], children: list[ChildChunk], doc_title: str):

    DIVIDER = "═" * 70

    # ── CONSOLE: parent chunks ────────────────────────────────────────────
    print(f"\n{DIVIDER}")
    print(f"  DOCUMENT: {doc_title}")
    print(f"  {len(parents)} parent chunks  |  {len(children)} child chunks")
    print(DIVIDER)

    for p in parents:
        path_str = " > ".join(p.heading_path) if p.heading_path else "—"
        print(f"\n{'─'*70}")
        print(f"[{p.parent_id}]  📍 {path_str}")
        print(f"  tokens: {p.token_count}  |  page: {p.page_start or 'N/A'}")
        print(f"{'─'*70}")
        print(p.markdown)

    # ── FILE: parent chunks ───────────────────────────────────────────────
    with open(OUTPUT_PARENTS, "w", encoding="utf-8") as f:
        f.write(f"DOCUMENT: {doc_title}\n")
        f.write(f"{len(parents)} parent chunks\n\n")
        for p in parents:
            path_str = " > ".join(p.heading_path) if p.heading_path else "—"
            f.write(f"\n{'─'*70}\n")
            f.write(f"[{p.parent_id}]\n")
            f.write(f"Heading path : {path_str}\n")
            f.write(f"Tokens       : {p.token_count}\n")
            f.write(f"Page         : {p.page_start or 'N/A'}\n")
            f.write(f"{'─'*70}\n")
            f.write(p.markdown + "\n")

    print(f"\n✅  Parents written → {OUTPUT_PARENTS}")

    # ── FILE: child chunks ────────────────────────────────────────────────
    with open(OUTPUT_CHILDREN, "w", encoding="utf-8") as f:
        f.write(f"DOCUMENT: {doc_title}\n")
        f.write(f"{len(children)} child chunks\n\n")
        for c in children:
            path_str = " > ".join(c.heading_path) if c.heading_path else "—"
            f.write(f"\n{'─'*70}\n")
            f.write(f"[{c.child_id}]  →  parent: {c.parent_id}\n")
            f.write(f"Heading path : {path_str}\n")
            f.write(f"Tokens       : {c.token_count}\n")
            f.write(f"{'─'*70}\n")
            f.write(c.markdown + "\n")

    print(f"✅  Children written → {OUTPUT_CHILDREN}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def ingest(file_path: str):
    print(f"\n📂  File     : {file_path}")

    fmt = detect_format(file_path)
    print(f"🔍  Format   : {fmt.upper()}")

    print(f"⚙️   Engine   : {'docx parser' if fmt == 'docx' else 'PyMuPDF'}")
    elements = extract_docx(file_path) if fmt == "docx" else extract_pdf(file_path)
    print(f"📄  Elements : {len(elements)} extracted")

    doc_title = extract_title(elements)
    print(f"📌  Title    : {doc_title}")

    parents  = build_parent_chunks(elements, doc_title)
    children = build_child_chunks(parents)

    print(f"🗂️   Parents  : {len(parents)} chunks (max {PARENT_MAX_TOKENS} tokens each)")
    print(f"🔹  Children : {len(children)} chunks (max {CHILD_MAX_TOKENS} tokens each)")

    print_and_save(parents, children, doc_title)


# ── SET YOUR FILE PATH HERE ──────────────────────────────────────────────────
FILE_PATH = r"D:\-- Ayush Rana --\archelon\backend\archelon_testing\Ayush_Rana_AI_and_Automation_Developer_Resume.pdf"
# ────────────────────────────────────────────────────────────────────────────

ingest(FILE_PATH)


📂  File     : D:\-- Ayush Rana --\archelon\backend\archelon_testing\Ayush_Rana_AI_and_Automation_Developer_Resume.pdf
🔍  Format   : PDF
⚙️   Engine   : PyMuPDF
📄  Elements : 115 extracted
📌  Title    : Ayush Rana
🗂️   Parents  : 46 chunks (max 800 tokens each)
🔹  Children : 48 chunks (max 150 tokens each)

══════════════════════════════════════════════════════════════════════
  DOCUMENT: Ayush Rana
  46 parent chunks  |  48 child chunks
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
[parent_0000]  📍 Ayush Rana
  tokens: 76  |  page: 1
──────────────────────────────────────────────────────────────────────
# Ayush Rana

Borivali (W), Mumbai, Maharashtra | Open to relocation (Pune)

� 9082038540

� ayushrana193@gmail.com

� linkedin.com/in/ayushrana12

Automation and Backend Developer actively building into Agentic AI - production experience in LLM workflows, n8n

automation, and Python-based 